In [11]:
# ------------------- CREATE eICU ANEMIA VALIDATION DATASET -------------------
import pandas as pd
import numpy as np
import re
import os

# Load eICU data
eicu_path = r'.........eicu-collaborative-research-database-2.0'

# Load diagnosis data
diagnosis_eicu = pd.read_csv(os.path.join(eicu_path, 'diagnosis.csv.gz'), 
                             low_memory=False)
print(f"diagnosis.csv loaded: {diagnosis_eicu.shape}")

# Load medication data
medication_eicu = pd.read_csv(os.path.join(eicu_path, 'medication.csv.gz'),
                              low_memory=False)
print(f"medication.csv loaded: {medication_eicu.shape}")

# Load admission drugs (alternative source for medications)
admission_drugs = pd.read_csv(os.path.join(eicu_path, 'admissionDrug.csv.gz'),
                              low_memory=False)
print(f"admissionDrug.csv loaded: {admission_drugs.shape}")

# -------------------- Clean ICD9 codes (handle mixed formats) --------------------
def extract_icd9_code(icd9_value):
    """Extract ICD9 code from eICU format (e.g., '414.00, I25.10' -> '414')"""
    if pd.isna(icd9_value):
        return None
    # Convert to string
    icd9_str = str(icd9_value)
    # Take first code before comma or space
    icd9_str = icd9_str.split(',')[0].split()[0].strip()
    # Remove any decimal points for prefix matching
    icd9_str = icd9_str.split('.')[0]
    # Remove non-numeric characters
    icd9_str = re.sub(r'[^0-9]', '', icd9_str)
    return icd9_str if len(icd9_str) >= 3 else None

diagnosis_eicu['ICD9_CODE'] = diagnosis_eicu['icd9code'].apply(extract_icd9_code)
diagnosis_eicu = diagnosis_eicu.dropna(subset=['ICD9_CODE'])

# -------------------- Get ANEMIA patients from eICU --------------------
anemia_icd9_prefixes = ['280', '281', '282', '283', '284', '285']

# Filter anemia patients
diagnoses_anemia_eicu = diagnosis_eicu[
    diagnosis_eicu['ICD9_CODE'].str[:3].isin(anemia_icd9_prefixes)
]
patients_anemia_eicu = set(diagnoses_anemia_eicu['patientunitstayid'].unique())
print(f"Anemia patients in eICU: {len(patients_anemia_eicu)}")

# -------------------- Process medication data for anemia patients --------------------
# Clean drug names function
def clean_drug_name_eicu(name):
    if pd.isnull(name):
        return ""
    name = str(name).lower().strip()
    # Remove dosage information
    name = re.sub(r'\b\d+(\.\d+)?\s*(mg|ml|mcg|g|mcg|tablet|tab|capsule|cap|inj|solution|suspension)\b', '', name)
    name = re.sub(r'\b\d+\s*(%|mg/ml|mcg/ml)\b', '', name)
    # Remove common suffixes
    name = re.sub(r'\s*(tab|caps|inj|soln|sol|susp)\s*', ' ', name)
    # Remove punctuation
    name = re.sub(r'[^\w\s]', ' ', name)
    # Remove extra spaces
    name = re.sub(r'\s+', ' ', name)
    # Remove common dosing schedules
    name = re.sub(r'\b(q\d+[h]?|bid|tid|qid|qd|prn|daily|hour|weekly|monthly)\b', '', name)
    return name.strip()

def get_patient_medications(patient_ids):
    """Get all medications for given patients from medication and admissionDrug tables"""
    
    # Filter medication table
    med_filtered = medication_eicu[medication_eicu['patientunitstayid'].isin(patient_ids)]
    
    # Extract drug names from medication table
    medications = []
    
    # From medication.csv (administered during stay)
    if len(med_filtered) > 0:
        for _, row in med_filtered.iterrows():
            drug_name = row.get('drugname', '')
            if pd.notna(drug_name) and str(drug_name).strip():
                medications.append({
                    'patientunitstayid': row['patientunitstayid'],
                    'drug_name': clean_drug_name_eicu(drug_name)
                })
    
    # From admissionDrug.csv (medications at admission)
    adm_filtered = admission_drugs[admission_drugs['patientunitstayid'].isin(patient_ids)]
    if len(adm_filtered) > 0:
        for _, row in adm_filtered.iterrows():
            # Try different column names for drug name in admissionDrug
            drug_name = row.get('drugname', row.get('medicationname', row.get('drug', '')))
            if pd.notna(drug_name) and str(drug_name).strip():
                medications.append({
                    'patientunitstayid': row['patientunitstayid'],
                    'drug_name': clean_drug_name_eicu(drug_name)
                })
    
    return pd.DataFrame(medications)

# Get medications for anemia patients
print("\nExtracting medications for anemia patients...")
medications_df = get_patient_medications(patients_anemia_eicu)
print(f"Total medication records: {len(medications_df):,}")

# Remove duplicates (same patient, same drug)
medications_df = medications_df.drop_duplicates(subset=['patientunitstayid', 'drug_name'])
print(f"After removing duplicates: {len(medications_df):,}")

# Filter out empty drug names
medications_df = medications_df[medications_df['drug_name'].str.len() > 0]
print(f"After filtering empty names: {len(medications_df):,}")

# -------------------- Create user-drug interaction matrix --------------------
# Group by patient and drug to create ratings
user_drug_eicu = medications_df.groupby(['patientunitstayid', 'drug_name']).size().reset_index(name='rating')
user_drug_eicu['rating'] = 1  # Binarize for implicit feedback

# Rename columns to match your anemia format
user_drug_eicu = user_drug_eicu.rename(columns={
    'patientunitstayid': 'user',
    'drug_name': 'item'
})

# -------------------- Filter low-frequency items (optional) --------------------
MIN_DRUG_FREQUENCY = 5
drug_counts = user_drug_eicu['item'].value_counts()
frequent_drugs = drug_counts[drug_counts >= MIN_DRUG_FREQUENCY].index
user_drug_eicu = user_drug_eicu[user_drug_eicu['item'].isin(frequent_drugs)]
print(f"After filtering drugs with <{MIN_DRUG_FREQUENCY} occurrences: {user_drug_eicu['item'].nunique()} unique drugs")

# Filter patients with at least 2 drugs
patient_counts = user_drug_eicu['user'].value_counts()
active_patients = patient_counts[patient_counts >= 2].index
user_drug_eicu = user_drug_eicu[user_drug_eicu['user'].isin(active_patients)]
print(f"After filtering patients with <2 drugs: {user_drug_eicu['user'].nunique()} patients")

# -------------------- Print statistics --------------------
print("\n" + "="*70)
print("eICU ANEMIA DATASET STATISTICS")
print("="*70)
print(f"Total anemia patients: {user_drug_eicu['user'].nunique():,}")
print(f"Total interactions: {len(user_drug_eicu):,}")
print(f"Unique drugs: {user_drug_eicu['item'].nunique():,}")
print(f"Average drugs per patient: {len(user_drug_eicu) / user_drug_eicu['user'].nunique():.2f}")
print(f"Interaction density: {len(user_drug_eicu) / (user_drug_eicu['user'].nunique() * user_drug_eicu['item'].nunique()):.4f}")

print("\nFirst 10 rows:")
print(user_drug_eicu.head(10))

# -------------------- Save to CSV --------------------
output_path = r'............user_drug_rating_visit_eicu_anemia.csv'
user_drug_eicu.to_csv(output_path, index=False)
print(f"\n✓ Saved: {output_path}")

# -------------------- Compare with MIMIC-III anemia dataset --------------------
print("\n" + "="*70)
print("COMPARISON WITH MIMIC-III ANEMIA DATASET")
print("="*70)

# Load MIMIC-III anemia dataset for comparison
mimic_anemia_path = r'...........user_drug_rating_visit_anemia.csv'
mimic_anemia_df = pd.read_csv(mimic_anemia_path)

print(f"\n{'Metric':<35} {'MIMIC-III Anemia':<20} {'eICU Anemia':<15}")
print("-" * 70)
print(f"{'Patients':<35} {mimic_anemia_df['user'].nunique():<20,} {user_drug_eicu['user'].nunique():<15,}")
print(f"{'Interactions':<35} {len(mimic_anemia_df):<20,} {len(user_drug_eicu):<15,}")
print(f"{'Unique Drugs':<35} {mimic_anemia_df['item'].nunique():<20,} {user_drug_eicu['item'].nunique():<15,}")
print(f"{'Avg Drugs/Patient':<35} {len(mimic_anemia_df)/mimic_anemia_df['user'].nunique():<20.2f} {len(user_drug_eicu)/user_drug_eicu['user'].nunique():<15.2f}")

# Drug overlap analysis
mimic_drugs = set(mimic_anemia_df['item'].str.lower().unique())
eicu_drugs = set(user_drug_eicu['item'].str.lower().unique())
overlap = mimic_drugs.intersection(eicu_drugs)
overlap_percent_mimic = 100 * len(overlap) / len(mimic_drugs) if len(mimic_drugs) > 0 else 0
overlap_percent_eicu = 100 * len(overlap) / len(eicu_drugs) if len(eicu_drugs) > 0 else 0

print(f"\n{'Drug Overlap Size':<35} {len(mimic_drugs):<20,} {len(eicu_drugs):<15,}")
print(f"{'Common Drugs':<35} {len(overlap):<20,} -")
print(f"{'Overlap % of MIMIC Drugs':<35} {overlap_percent_mimic:<20.1f}%")
print(f"{'Overlap % of eICU Drugs':<35} {overlap_percent_eicu:<20.1f}%")

# Top 20 common drugs
print("\nTop 20 Common Drugs between MIMIC-III and eICU:")
common_drugs_list = list(overlap)[:20]
for i, drug in enumerate(sorted(common_drugs_list), 1):
    print(f"  {i:2d}. {drug}")

# -------------------- Quality checks --------------------
print("\n" + "="*70)
print("QUALITY CHECKS")
print("="*70)

# Check for missing values
print(f"\nMissing values:")
print(f"  Users: {user_drug_eicu['user'].isna().sum()}")
print(f"  Items: {user_drug_eicu['item'].isna().sum()}")

# Distribution of interactions per patient
interactions_per_patient = user_drug_eicu.groupby('user').size()
print(f"\nInteractions per patient:")
print(f"  Min: {interactions_per_patient.min()}")
print(f"  Max: {interactions_per_patient.max()}")
print(f"  Mean: {interactions_per_patient.mean():.1f}")
print(f"  Median: {interactions_per_patient.median():.1f}")
print(f"  Quartiles: Q1={interactions_per_patient.quantile(0.25):.1f}, Q3={interactions_per_patient.quantile(0.75):.1f}")

# Top 20 drugs in eICU anemia dataset
top_drugs_eicu = user_drug_eicu['item'].value_counts().head(20)
print(f"\nTop 20 drugs in eICU anemia dataset:")
for i, (drug, count) in enumerate(top_drugs_eicu.items(), 1):
    print(f"  {i:2d}. {drug}: {count:,} patients")

# Patient overlap between MIMIC and eICU (should be 0 since different databases)
mimic_patients = set(mimic_anemia_df['user'].astype(str))
eicu_patients = set(user_drug_eicu['user'].astype(str))
patient_overlap = mimic_patients.intersection(eicu_patients)

print(f"\nPatient overlap between datasets:")
print(f"  Overlap size: {len(patient_overlap)} (expected 0 - different databases)")
print(f"  MIMIC patients: {len(mimic_patients):,}")
print(f"  eICU patients: {len(eicu_patients):,}")

print("\n" + "="*70)
print("✅ eICU ANEMIA VALIDATION DATASET CREATED SUCCESSFULLY")
print("="*70)

# -------------------- Optional: Export summary statistics --------------------
summary_stats = pd.DataFrame({
    'Database': ['MIMIC-III', 'eICU'],
    'Patients': [mimic_anemia_df['user'].nunique(), user_drug_eicu['user'].nunique()],
    'Interactions': [len(mimic_anemia_df), len(user_drug_eicu)],
    'Unique_Drugs': [mimic_anemia_df['item'].nunique(), user_drug_eicu['item'].nunique()],
    'Avg_Drugs_Per_Patient': [
        len(mimic_anemia_df) / mimic_anemia_df['user'].nunique(),
        len(user_drug_eicu) / user_drug_eicu['user'].nunique()
    ]
})

summary_stats.to_csv(r'............anemia_dataset_comparison.csv', index=False)
print("\n✓ Summary statistics saved to 'anemia_dataset_comparison.csv'")

diagnosis.csv loaded: (2710672, 7)
medication.csv loaded: (7301853, 15)
admissionDrug.csv loaded: (874920, 14)
Anemia patients in eICU: 5198

Extracting medications for anemia patients...
Total medication records: 364,745
After removing duplicates: 102,303
After filtering empty names: 102,303
After filtering drugs with <5 occurrences: 1347 unique drugs
After filtering patients with <2 drugs: 4432 patients

eICU ANEMIA DATASET STATISTICS
Total anemia patients: 4,432
Total interactions: 100,281
Unique drugs: 1,347
Average drugs per patient: 22.63
Interaction density: 0.0168

First 10 rows:
     user                              item  rating
0  145270                acetaminophen po s       1
1  145270           calcium gluconate 10 iv       1
2  145270   chlorhexidine gluconate 0 12 mt       1
3  145270                   famotidine po s       1
4  145270            fentanyl citrate ml ij       1
5  145270  flex cont magnesium sulfate 4 ij       1
6  145270                  furosemide ml 